# sep05 — seeds 2+3 of the leak-free rerun, plus the leaked-SFT cell (Kaggle T4 x2)

**Settings → Accelerator → GPU T4 x2, Internet → On.** Secrets: `HF_TOKEN` (required),
`WANDB_API_KEY` (optional → offline fallback). Budget ~10–10.5h of the 12h cap; every
phase is idempotent and checkpoints sync to per-seed Hub repos, so a killed session
resumes: just rerun the notebook.

**What this answers, and for whom** (round-3 panel):
- *Two more leak-free seeds* (Reviewer 2 → accept): is seed 0's 148/648 endpoint a
  draw or the recipe? Trained in PARALLEL, one per T4, `save_steps: 10` so the
  110/120/130/140 window three reviewers have asked about is checkpointed at last.
- *Leaked-SFT* (Reviewer 4 → accept): the missing {leaked, clean} x {SFT, GRPO} cell.
  SFT trained on ANSWER-ORDERED prompts, evaluated on the same shuffled test split.

**Prespecified readings** (written before launch):

| Result | Reading |
|---|---|
| seeds 1+2 endpoints high (same order as 148/648), no collapse anywhere incl. 100–150 window | the repaired recipe is stable; seed-0 was not a draw |
| a seed collapses | report it; the §8 claim becomes seed-dependent and §12 says so |
| leaked-SFT: high copy rate on shuffled test, groups near the 1.4-slot floor | leakage dominates regardless of RL: the RL attribution weakens and the paper says so |
| leaked-SFT: copy rate low, groups ≈ clean SFT (~0.32–0.33) | SFT under the leak *suppresses* copying (as clean SFT does, 0.3%): GRPO manufactured the copier its own init suppressed — an RL-specific amplification result |
| leaked-SFT intermediate | report as measured; both framings survive with caveats |

Phases: setup → configs+read-backs → leaked-SFT data (no GPU) → seeds train in
parallel (~6h) → leaked-SFT train (~1h) → per-seed val curves incl. dense window
(~1.7h) → one 7-arm test session (~1.4h) → copy-rule + verdicts → persist.


In [ ]:
# Cell 1 — setup. trl pinned to 1.10.0 (the reported runs' trainer); vllm deferred.
import os, glob, json, subprocess, time
T0 = time.time()
def elapsed(): print(f'[budget] {(time.time()-T0)/3600:.2f}h elapsed')
from kaggle_secrets import UserSecretsClient
S = UserSecretsClient()
os.environ['HF_TOKEN'] = S.get_secret('HF_TOKEN')
try: os.environ['WANDB_API_KEY'] = S.get_secret('WANDB_API_KEY')
except Exception: os.environ['WANDB_MODE'] = 'offline'; print('no WANDB secret -> offline')
HF_USER = 'jacksonlukas'

!git clone -b analysis/aug21 https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip uninstall -y -q vllm 2>/dev/null || true
!pip install -q -e . openai peft trl==1.10.0 bitsandbytes accelerate
r = subprocess.run(['python','-c',
    'from trl import GRPOConfig, GRPOTrainer; from trl import SFTConfig, SFTTrainer; import trl; print("trl", trl.__version__, "trainers import OK")'],
    capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'trainer preflight FAILED'
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!make data
r = subprocess.run(['python','-c',(
    'from connections_rl.data.loader import load_puzzles;'
    'print(len(load_puzzles("data/splits/puzzles_test.json")))')], capture_output=True, text=True)
assert int(r.stdout.strip()) == 162, 'test split is not 162'
print('setup OK'); elapsed()


In [ ]:
# Cell 2 — per-seed configs + prompt read-back + SFT init adapter.
import yaml, random
from huggingface_hub import snapshot_download
base = yaml.safe_load(open('configs/train/grpo-7b-shuffled.yaml'))
SEEDS = [1, 2]
os.makedirs('results-analysis/sep05', exist_ok=True)
for s in SEEDS:
    cfg = dict(base)
    cfg['seed'] = s
    cfg['output_dir'] = f'artifacts/grpo-7b-shuffled-s{s}'
    cfg['ckpt_hub_repo'] = f'connections-rl-grpo-7b-shuffled-s{s}-ckpt'
    cfg['run_name'] = f'connections-rl-grpo-qwen7b-shuffled-s{s}'
    # THE config line three rounds of reviewers asked for: checkpoint the
    # 100->150 window (and everything else) every 10 steps.
    cfg['save_steps'] = 10
    yaml.safe_dump(cfg, open(f'results-analysis/sep05/grpo-7b-shuffled-s{s}.yaml','w'), sort_keys=False)
    print(f'wrote config for seed {s}')

# Read-back: prompts shuffled, deterministic, complete (the standing guard).
check = '''
from connections_rl.train.grpo import build_dataset, load_puzzle_split
recs = load_puzzle_split('data/splits', 'train')[:5]
ds1, ds2 = build_dataset(recs), build_dataset(recs)
for i, rec in enumerate(recs):
    answer_order = [w.upper() for g in rec['answers'] for w in g['members']]
    def words_of(ds):
        u = ds[i]['prompt'][1]['content']
        return [w.strip().upper() for w in u.replace('Words:', '', 1).split(',')]
    w1, w2 = words_of(ds1), words_of(ds2)
    assert w1 == w2, 'not deterministic'
    assert sorted(w1) == sorted(answer_order), 'not complete'
    assert w1 != answer_order, 'STILL ANSWER-ORDERED'
print('read-back OK: shuffled, deterministic, complete on 5 records')
'''
r = subprocess.run(['python','-c',check], capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'read-back FAILED -- stop'

snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='artifacts/sft-7b',
                  token=os.environ['HF_TOKEN'])
assert os.path.exists('artifacts/sft-7b/adapter_config.json'), 'SFT init adapter incomplete'
print('configs + read-back + init adapter OK'); elapsed()


In [ ]:
# Cell 3 — leaked-SFT data: rebuild each SFT user turn in ANSWER-KEY order (no GPU).
# The answer key is recovered from the example's own assistant target, so this
# mirrors the GRPO bug's presentation exactly: positions 1-4 the first group, etc.
import re as _re
def parse_answer_groups(text):
    m = _re.search(r'<ANSWER>(.*?)</ANSWER>', text, _re.S)
    assert m, 'assistant target has no ANSWER block'
    gs = []
    for line in m.group(1).strip().splitlines():
        gm = _re.match(r'\s*Group \d+:\s*(.+)', line)
        if gm: gs.append([w.strip() for w in gm.group(1).split(',')])
    assert len(gs) == 4 and all(len(g) == 4 for g in gs), 'target is not 4x4'
    return gs

n = changed = 0
with open('data/splits/train.jsonl') as f_in, open('data/splits/train_leaked.jsonl','w') as f_out:
    for line in f_in:
        row = json.loads(line)
        msgs = row['messages']
        groups = parse_answer_groups(msgs[-1]['content'])
        leaked_words = [w for g in groups for w in g]
        old_user = msgs[1]['content']
        old_words = [w.strip() for w in old_user.replace('Words:','',1).split(',')]
        assert sorted(w.upper() for w in old_words) == sorted(w.upper() for w in leaked_words), \
            f'word multiset mismatch on puzzle {row.get("puzzle_id")}'
        new_user = 'Words: ' + ', '.join(leaked_words)
        changed += (new_user != old_user)
        msgs[1] = dict(msgs[1], content=new_user)
        f_out.write(json.dumps(row) + '\n')
        n += 1
print(f'{n} rows; {changed} differ from shuffled presentation')
assert n > 0 and changed / n > 0.95, 'leaked data barely differs -- inspect'

# READ-BACK on the constructed file: every quadruple must be an answer group.
checked = 0
for line in open('data/splits/train_leaked.jsonl'):
    row = json.loads(line)
    words = [w.strip() for w in row['messages'][1]['content'].replace('Words:','',1).split(',')]
    keys = [frozenset(g) for g in parse_answer_groups(row['messages'][-1]['content'])]
    assert len(words) == 16
    for i in range(0, 16, 4):
        assert frozenset(words[i:i+4]) in keys, 'quadruple is not an answer group'
    checked += 1
    if checked >= 50: break
print('leaked-SFT data read-back OK on', checked, 'rows')

sft_base = yaml.safe_load(open('configs/train/sft-7b.yaml'))
sft_cfg = dict(sft_base)
sft_cfg['train_data'] = 'data/splits/train_leaked.jsonl'
sft_cfg['output_dir'] = 'artifacts/sft-7b-leaked'
sft_cfg['run_name'] = 'connections-rl-sft-qwen7b-leaked'
yaml.safe_dump(sft_cfg, open('results-analysis/sep05/sft-7b-leaked.yaml','w'), sort_keys=False)
print('leaked-SFT config written'); elapsed()


In [ ]:
# Cell 4 — train seeds 1 and 2 IN PARALLEL, one per T4 (~6h incl. Hub syncs).
# resume_from_hub makes this cell safe to rerun after a session death.
procs = {}
for s, gpu in [(1,'0'), (2,'1')]:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
    logf = open(f'/kaggle/working/train-s{s}.log','w')
    procs[s] = (subprocess.Popen(['python','-m','connections_rl.train.grpo',
                 '--config', f'results-analysis/sep05/grpo-7b-shuffled-s{s}.yaml'],
                 env=env, stdout=logf, stderr=subprocess.STDOUT), logf)
    print(f'seed {s} launched on GPU {gpu}')
for s, (p, logf) in procs.items():
    rc = p.wait(); logf.close()
    print(f'seed {s} training exit: {rc}')
    !tail -5 /kaggle/working/train-s{s}.log
for s in [1, 2]:
    ckpts = sorted(glob.glob(f'artifacts/grpo-7b-shuffled-s{s}/checkpoint-*'),
                   key=lambda x: int(x.rsplit('-',1)[1]))
    steps = [int(c.rsplit('-',1)[1]) for c in ckpts]
    print(f'seed {s}: {len(steps)} checkpoints, window ckpts:',
          [t for t in steps if 100 < t < 150])
    assert steps and max(steps) >= 400, f'seed {s} incomplete -- rerun this cell (resumes from Hub)'
    for want in (110, 120, 130, 140):
        assert want in steps, f'seed {s} missing dense checkpoint {want}'
print('both seeds trained, dense window checkpointed'); elapsed()


In [ ]:
# Cell 5 — train leaked-SFT (~1h, GPU 0), then push the adapter to the Hub.
if not os.path.exists('artifacts/sft-7b-leaked/adapter_config.json'):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
    r = subprocess.run(['python','-m','connections_rl.train.sft',
                        '--config','results-analysis/sep05/sft-7b-leaked.yaml'], env=env)
    print('leaked-SFT training exit:', r.returncode)
assert os.path.exists('artifacts/sft-7b-leaked/adapter_config.json'), 'leaked-SFT adapter missing'
from huggingface_hub import HfApi
try:
    api = HfApi(token=os.environ['HF_TOKEN'])
    repo = f'{HF_USER}/connections-rl-sft-7b-leaked'
    if not api.repo_exists(repo): api.create_repo(repo, private=True)
    api.upload_folder(folder_path='artifacts/sft-7b-leaked', repo_id=repo)
    print('leaked-SFT adapter -> Hub')
except Exception as e:
    print('Hub upload of leaked-SFT failed (continuing; adapter is in the zip):', e)
print('leaked-SFT ready'); elapsed()


In [ ]:
# Cell 6 — vllm now (training done); per-seed val curves incl. the dense window.
!pip install -q vllm
!pip show vllm | grep -E '^(Name|Version)' | tee results-analysis/sep05/session-versions.txt
import urllib.request
CURVE_STEPS = [50, 100, 110, 120, 130, 140, 150, 200, 250, 300, 350, 400, 403]
def serve(mods):
    proc = subprocess.Popen(
        'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
        '--enable-lora --enforce-eager --max-lora-rank 16 --max-model-len 2048 '
        '--gpu-memory-utilization 0.85 --lora-modules ' + ' '.join(mods),
        shell=True, stdout=open('/kaggle/working/vllm.log','a'), stderr=subprocess.STDOUT)
    for _ in range(150):
        try: urllib.request.urlopen('http://localhost:8000/health'); print('vLLM ready'); return proc
        except Exception: time.sleep(10)
    raise RuntimeError('vLLM failed -- see /kaggle/working/vllm.log')
def kill(proc):
    !pkill -f 'vllm serve' 2>/dev/null || true
    proc.wait(); time.sleep(30)

PEAK = {}
for s in [1, 2]:
    out = f'results-analysis/sep05/ckpt-curve-7b-shuffled-s{s}'
    if not os.path.exists(out + '.json'):
        mods = [f's{s}-ckpt-{t}=artifacts/grpo-7b-shuffled-s{s}/checkpoint-{t}' for t in CURVE_STEPS]
        proc = serve(mods)
        arms_arg = ','.join(f's{s}-ckpt-{t}:{t}' for t in CURVE_STEPS)
        r = subprocess.run(['python','-m','connections_rl.eval.checkpoint_curve',
                            '--arms', arms_arg, '--puzzles','data/splits/puzzles_val.json',
                            '--out', out])
        kill(proc)
        assert r.returncode == 0, f'curve failed for seed {s}'
    pts = json.load(open(out + '.json'))
    peak = max(pts, key=lambda p: p['semantic_groups_correct'])
    PEAK[s] = peak['step']
    print(f'seed {s} val peak: step {PEAK[s]} | curve:',
          [(p['step'], round(p['semantic_groups_correct'],4)) for p in pts])
    win = [(p['step'], round(p['semantic_groups_correct'],4)) for p in pts if 100 <= p['step'] <= 150]
    print(f'seed {s} 100-150 window:', win)
elapsed()


In [ ]:
# Cell 7 — ONE test session, seven arms, greedy, with capture.
ARMS = {'base': 'Qwen/Qwen2.5-7B-Instruct',
        'sft': 'connections-rl-sft-7b',
        'sft-leaked': 'connections-rl-sft-7b-leaked',
        's1-peak': f's1-ckpt-{PEAK[1]}', 's1-final': 's1-final',
        's2-peak': f's2-ckpt-{PEAK[2]}', 's2-final': 's2-final'}
mods = ['connections-rl-sft-7b=artifacts/sft-7b',
        'connections-rl-sft-7b-leaked=artifacts/sft-7b-leaked',
        f's1-ckpt-{PEAK[1]}=artifacts/grpo-7b-shuffled-s1/checkpoint-{PEAK[1]}',
        's1-final=artifacts/grpo-7b-shuffled-s1',
        f's2-ckpt-{PEAK[2]}=artifacts/grpo-7b-shuffled-s2/checkpoint-{PEAK[2]}',
        's2-final=artifacts/grpo-7b-shuffled-s2']
proc = serve(mods)
lines = ['puzzles: data/splits/puzzles_test.json',
         'out_dir: results-analysis/sep05/sep05-session-test',
         'n_resamples: 1000', 'seed: 0', 'capture_generations: true', '', 'arms:']
for a, m in ARMS.items():
    lines += [f'  - name: {a}', f'    model: {m}', '    temperature: 0.0']
open('results-analysis/sep05/sep05_test.yaml','w').write('\n'.join(lines) + '\n')
r = subprocess.run(['python','-m','connections_rl.eval.run',
                    '--config','results-analysis/sep05/sep05_test.yaml'])
assert r.returncode == 0, 'test eval failed'
kill(proc)
print('7-arm session done'); elapsed()


In [ ]:
# Cell 8 — copy-rule test + verdicts against the prespecified table.
import re as _re
def parse_groups(text):
    m = _re.search(r'<ANSWER>(.*?)</ANSWER>', text, _re.S)
    if not m: return None
    gs = [[w.strip().upper() for w in gm.group(1).split(',')]
          for line in m.group(1).strip().splitlines()
          if (gm := _re.match(r'\s*Group \d+:\s*(.+)', line))]
    return gs if len(gs) == 4 else None
def prompt_words(pf):
    chat = json.loads(pf) if isinstance(pf, str) else pf
    u = next(m['content'] for m in chat if m['role'] == 'user')
    return [w.strip().upper() for w in u.replace('Words:', '', 1).split(',')]

summary = {'session': 'sep05 session', 'decoding': 'greedy T=0.0',
           'peaks': PEAK, 'arms': {}}
hdr = f"{'arm':<12} {'groups(0-4)':<24} {'slots':<9} {'invalid':<9} {'reward':<9} {'copy g/resp'}"
print(hdr); print('-'*len(hdr))
for arm in ARMS:
    d = f'results-analysis/sep05/sep05-session-test/{arm}'
    m = json.load(open(f'{d}/metrics.json')); o = m['summary']['OVERALL']
    quad = tot = pure = parsed = 0
    for line in open(f'{d}/generations.jsonl'):
        r = json.loads(line)
        w = prompt_words(r['prompt']); gs = parse_groups(r['generation'])
        if gs is None or len(w) != 16: continue
        parsed += 1
        quads = [set(w[i:i+4]) for i in range(0, 16, 4)]
        h = sum(1 for g in gs if set(g) in quads)
        quad += h; tot += 4; pure += (h == 4)
    g, inv, rw = o['groups_correct'], o['invalid_rate'], o['reward']
    row = {'groups_mean_ci': g, 'slots': round(g[0]*162), 'invalid_ci': inv,
           'reward_ci': rw, 'copy_group_rate': quad/tot if tot else None,
           'pure_copy_rate': pure/parsed if parsed else None, 'parsed': parsed}
    summary['arms'][arm] = row
    print(f"{arm:<12} {g[0]:.4f} [{g[1]:.3f},{g[2]:.3f}]   {round(g[0]*162):<9} "
          f"{inv[0]:<9.3f} {rw[0]:<9.4f} {quad/tot:.3f}/{pure/parsed:.3f}")

print()
print('Anchors: clean-SFT 0.32-0.33 across sessions; seed-0 shuffled-final 148/648;')
print('leaked-GRPO final 4/648 with copy 0.924; chance floor 0.0088 groups/puzzle.')
sl = summary['arms']['sft-leaked']
if sl['copy_group_rate'] > 0.5 and sl['groups_mean_ci'][0] < 3*0.0088*4:
    print('LEAKED-SFT VERDICT: collapses to a copier -> leakage dominates; RL attribution weakens.')
elif sl['copy_group_rate'] < 0.05 and sl['groups_mean_ci'][0] > 0.25:
    print('LEAKED-SFT VERDICT: retains held-out grouping without copying ->')
    print('GRPO manufactured the copier its own initialization suppressed (RL-specific amplification).')
else:
    print('LEAKED-SFT VERDICT: intermediate -- report as measured, both framings carry caveats.')
for s in [1, 2]:
    fin = summary['arms'][f's{s}-final']
    print(f"seed {s} endpoint: {fin['slots']}/648 (seed 0: 148/648)")
json.dump(summary, open('results-analysis/sep05/sep05_summary.json','w'), indent=1)
print('wrote results-analysis/sep05/sep05_summary.json'); elapsed()


In [ ]:
# Cell 9 — persist. Adapters are on the Hub already (per-seed ckpt repos + leaked-SFT).
!zip -qr /kaggle/working/sep05-outputs.zip results-analysis/sep05
print('zip ready: /kaggle/working/sep05-outputs.zip')
try:
    from huggingface_hub import HfApi
    HfApi(token=os.environ['HF_TOKEN']).upload_folder(
        folder_path='results-analysis/sep05',
        repo_id=f'{HF_USER}/connections-rl-results', repo_type='dataset',
        path_in_repo='sep05')
    print('Hub upload OK -> connections-rl-results/sep05')
except Exception as e:
    print('Hub upload skipped/failed (fine -- use the zip):', e)
elapsed()
